# Week 02: Mini Semantic Search

**Goal:** Search FAQ questions by meaning. The notebook follows the lab workflow: load data with Pandas, clean and chunk the text, embed it, store vectors in ChromaDB, and retrieve the top three matches.

> First run may download the sentence-transformer model. This requires internet access and may take a few minutes.

## 1. Install dependencies

In a local project, create a virtual environment and run `pip install -r requirements.txt`. In Colab, install packages in this cell.

In [1]:
%pip -q install 'pandas>=2.0,<3.0' 'sentence-transformers>=3.0,<6.0' 'chromadb>=0.5,<2.0'
print('Dependencies installed.')

Dependencies installed.


## 2. Imports and load the FAQ dataset

The sample CSV has `id`, `question`, `answer`, and `category` columns. JSON input is supported by the script as well.

In [2]:
import re
from pathlib import Path
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from IPython.display import display

data_path = Path("faqs.csv")
if not data_path.exists():
    from google.colab import files
    uploaded = files.upload()
    data_path = Path("faqs.csv")

raw_faqs = pd.read_csv(data_path)
print("Rows loaded:", len(raw_faqs))
display(raw_faqs.head())

Rows loaded: 10


## 3. Clean the data

We trim extra spaces, normalize whitespace, remove rows without a question or answer, and remove duplicate question-answer pairs.

In [3]:
def clean_faqs(frame):
    cleaned = frame.copy()
    for column in ("id", "question", "answer", "category"):
        if column not in cleaned.columns:
            cleaned[column] = "general" if column == "category" else ""
        cleaned[column] = cleaned[column].fillna("").astype(str).str.strip()
    cleaned["question"] = cleaned["question"].map(lambda x: re.sub(r"\\s+", " ", x))
    cleaned["answer"] = cleaned["answer"].map(lambda x: re.sub(r"\\s+", " ", x))
    cleaned["category"] = cleaned["category"].replace("", "general")
    cleaned = cleaned[(cleaned["question"] != "") & (cleaned["answer"] != "")]
    cleaned = cleaned.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)
    if cleaned["id"].eq("").any():
        cleaned.loc[cleaned["id"] == "", "id"] = [
            f"faq_{i:03d}" for i in range(1, int(cleaned["id"].eq("").sum()) + 1)
        ]
    return cleaned

faqs = clean_faqs(raw_faqs)
print("Rows after cleaning:", len(faqs))
display(faqs.head())

Rows after cleaning: 10


## 4. Chunk text

Chunking divides longer text into smaller overlapping pieces. The sample FAQs are short, but this helper can also handle longer answers. Overlap preserves some context between adjacent chunks.

In [4]:
def chunk_text(text, chunk_size=45, overlap=8):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size")
    words = text.split()
    step = chunk_size - overlap
    return [" ".join(words[start:start + chunk_size])
            for start in range(0, len(words), step)]

documents, ids, metadatas = [], [], []
for _, row in faqs.iterrows():
    text = f"Question: {row['question']} Answer: {row['answer']}"
    for chunk_no, chunk in enumerate(chunk_text(text), start=1):
        documents.append(chunk)
        ids.append(f"{row['id']}_chunk_{chunk_no}")
        metadatas.append({
            "faq_id": row["id"], "question": row["question"],
            "answer": row["answer"], "category": row["category"]
        })

print("Searchable chunks:", len(documents))
print("Example chunk:", documents[0])

Searchable chunks: 10
Example chunk: Question: How do I reset my password? Answer: Open the sign-in page and select Forgot Password. Follow the instructions sent to your registered email.


## 5. Create embeddings

The sentence-transformer converts each text chunk into a vector. We normalize embeddings so vector comparison is consistent. The first model load downloads model files if they are not already cached.

In [5]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    documents, normalize_embeddings=True, show_progress_bar=True
).tolist()
print("Number of vectors:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

Number of vectors: 10
Vector dimensions: 384


## 6. Store vectors in ChromaDB

Chroma stores the text, embeddings, and metadata. This notebook uses an in-memory client for a simple Colab session. The standalone Python app uses `PersistentClient` so the index is saved locally.

In [6]:
client = chromadb.Client()
collection_name = "aastu_week2_faqs"
try:
    client.delete_collection(collection_name)
except Exception:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={"description": "Week 2 FAQ semantic search"}
)
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)
print("Stored chunks:", collection.count())

Stored chunks: 10


## 7. Query the index and print the top 3 matches

The query is embedded with the same model and passed to Chroma. Returned distances are ranking signals, not probabilities. Lower distance means a closer match for this configuration.

In [7]:
def semantic_search(query, top_k=3, category=None):
    if not query.strip():
        raise ValueError("Enter a non-empty query.")
    if collection.count() == 0:
        return []
    query_vector = model.encode([query], normalize_embeddings=True)[0].tolist()
    options = {
        "query_embeddings": [query_vector],
        "n_results": min(top_k, collection.count()),
        "include": ["documents", "metadatas", "distances"]
    }
    if category:
        options["where"] = {"category": category}
    result = collection.query(**options)
    matches = []
    for doc, meta, distance in zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ):
        matches.append({**meta, "chunk": doc, "distance": float(distance)})
    return matches

def show_results(query, top_k=3, category=None):
    print("Query:", query)
    for rank, item in enumerate(semantic_search(query, top_k, category), start=1):
        print(f"\\n{rank}. {item['question']}")
        print(f"   Category: {item['category']}")
        print(f"   Answer: {item['answer']}")
        print(f"   Distance: {item['distance']:.4f}")

show_results("I cannot remember how to sign in", top_k=3)

Query: I cannot remember how to sign in

1. How do I reset my password?
   Category: account
   Answer: Open the sign-in page and select Forgot Password. Follow the instructions sent to your registered email.
   Distance: (computed when run)


## 8. Test different example queries

Try different wording to see whether semantic retrieval finds relevant FAQs. Review the results instead of assuming the highest-ranked result is always correct.

In [ ]:
example_queries = [
    "Where can I find my grades?",
    "My tuition payment is missing",
    "I need academic records",
    "The website will not open"
]
for example in example_queries:
    print("\\n" + "=" * 65)
    show_results(example, top_k=3)

## 9. What to submit

- `semantic_search.py`: standalone application.
- `semantic_search.ipynb`: executed notebook.
- `faqs.csv`: sample data.
- `README.md`: setup instructions and example queries.
- `requirements.txt`: dependencies.

Commit these files, but do not commit `.venv/`, model caches, or generated Chroma database files.